In [ ]:
%matplotlib widget
import os
import torch, einops
import matplotlib.pyplot as plt
import numpy as np

from util import configure_logging, download_and_concat, textfile_to_tokens_as_binary, get_batch
from tokenizer import train_bpe, Tokenizer
from transformer import Embedding, TransformerBlock, RMSNorm, Linear, MultiHeadAttention, TransformerLM
from optimizer import AdamW
configure_logging()  # or configure_logging(logging.DEBUG) for verbose

In [ ]:
# download some internet text
urls = [
    "https://gutenberg.org/cache/epub/1184/pg1184.txt",
    "https://gutenberg.org/cache/epub/1513/pg1513.txt",
]
download_and_concat(urls, "data/combined.txt", separator="\n<|endoftext|>\n")

In [ ]:
# tokenize this text
special_tokens = ["<|endoftext|>", "<|begin|>", "<|end|>"]
path = "../data/TinyStoriesV2-GPT4-valid.txt"
vocabulary_size = 5000

vocab, merges = train_bpe(path, vocabulary_size, special_tokens)
tokenizer = Tokenizer(vocab, merges, special_tokens)

In [ ]:
# convert the file into a raw-binary which we can read as mmmap.
textfile_to_tokens_as_binary("../data/TinyStoriesV2-GPT4-train.txt","data/train.bin", tokenizer, "wb")
textfile_to_tokens_as_binary("../data/TinyStoriesV2-GPT4-valid.txt","data/valid.bin", tokenizer, "wb")
#os.path.getsize("data/train.bin")
# data = np.memmap("data/train.bin", dtype=np.uint16, mode="r")
# get_batch(data, 20, 5) returns 20 batches of length 5 sequences inputs,labels

In [ ]:
train_data = np.memmap("words/train.bin", dtype=np.uint16, mode="r")
valid_data = np.memmap("words/train.bin", dtype=np.uint16, mode="r")

In [ ]:
#initialize the model
context_length = 256
d_model = 128
d_ff = 1344
num_heads = 16
num_layers = 4
rope_theta = 10000
model = TransformerLM(vocabulary_size, context_length, num_layers, d_model, d_ff, num_heads, rope_theta)

In [ ]:
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters())

In [ ]:
%matplotlib inline
from tqdm.auto import tqdm
from util import LiveLossPlot

batch_size = 64
iterations = 600
device = "mps"
val_every = 10  # how often to compute validation loss
model.to(device)

with LiveLossPlot(every=5) as plot:
    pbar = tqdm(range(iterations), desc="training")
    for iteration in pbar:
        model.train()
        inputs, targets = get_batch(train_data, batch_size, context_length, device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs.transpose(1, 2), targets)
        loss.backward()
        optimizer.step()

        plot.log(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

        if iteration % val_every == 0:
            model.eval()
            with torch.no_grad():
                vi, vt = get_batch(valid_data, batch_size, context_length, device)
                vloss = loss_function(model(vi).transpose(1, 2), vt)
            plot.log_val(vloss.item())

In [ ]:
import sys


def generate(prompt: str, max_tokens: int = 400, temperature: float = 0.9, width: int = 90) -> str:
    """Stream-decode tokens from `model` and print with word-wrap at `width`.

    Uses the surrounding notebook scope for `model`, `tokenizer`, `context_length`,
    and `device`. Stops early when the <|endoftext|> token is sampled. Returns the
    full generated string (prompt + completion) for any post-hoc use.
    """
    EOT_ID = tokenizer.encode("<|endoftext|>")[0]

    all_ids = tokenizer.encode(prompt)
    model_input = torch.tensor(all_ids, dtype=torch.long).unsqueeze(0).to(device)

    col = 0
    word_buf = ""

    def stream(text: str) -> None:
        nonlocal col, word_buf
        text = text.replace("\n", " ")
        for ch in text:
            if ch == " ":
                if word_buf:
                    space = 1 if col > 0 else 0
                    if col + space + len(word_buf) > width:
                        sys.stdout.write("\n")
                        col = 0
                    elif space:
                        sys.stdout.write(" ")
                        col += 1
                    sys.stdout.write(word_buf)
                    col += len(word_buf)
                    word_buf = ""
            else:
                word_buf += ch
        sys.stdout.flush()

    stream(prompt)
    printed = prompt

    model.eval()
    with torch.no_grad():
        for _ in range(max_tokens):
            logits = model(model_input)[:, -1]
            probs = torch.softmax(logits / temperature, dim=-1)
            pred = torch.multinomial(probs, num_samples=1)
            token_id = pred.item()
            if token_id == EOT_ID:
                break
            all_ids.append(token_id)
            model_input = torch.cat([model_input, pred], dim=1)[:, -context_length:]

            decoded = tokenizer.decode(all_ids)
            new_part = decoded[len(printed):]
            if new_part:
                stream(new_part)
                printed = decoded

    stream(" ")  # flush trailing partial word
    return tokenizer.decode(all_ids)

In [ ]:
generate("tell me a story", max_tokens=400, temperature=0.9);